<a href="https://colab.research.google.com/github/DanylchenkoKateryna/NLP-Lab-works/blob/main/notebooks/lab8_topic_modeling_lsa_lda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/DanylchenkoKateryna/NLP-Lab-works/blob/main/notebooks/lab8_topic_modeling_lsa_lda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 8 — Topic Modeling: LSA / LDA

**Corpus:** 20 Newsgroups — 3 classes: `alt.atheism`, `sci.electronics`, `soc.religion.christian`  
**Goal:** Build LSA and LDA topic models, interpret topics manually, identify weak/noisy topics, compare models.  
**Base data:** `data/processed_v2/processed_v2.csv` (cleaned, PII-masked text)


## 1. Install Dependencies

In [4]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "scikit-learn", "pandas", "numpy", "matplotlib"], check=True)
    print("Colab: dependencies installed.")
else:
    print("Local environment — dependencies assumed installed.")


Colab: dependencies installed.


In [5]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for compatibility
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD, LatentDirichletAllocation

print('Python:', sys.version)
import sklearn; print('scikit-learn:', sklearn.__version__)
print('numpy:', np.__version__)
print('pandas:', pd.__version__)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
scikit-learn: 1.6.1
numpy: 2.0.2
pandas: 2.2.2


## 2. Data Access

We load `processed_v2.csv` — the cleaned, PII-masked version of the corpus produced in Lab 2.  
Raw data is never modified.

In [6]:
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not Path("/content/NLP-Lab-works").exists():
        os.system("git clone https://github.com/DanylchenkoKateryna/NLP-Lab-works.git /content/NLP-Lab-works")
    ROOT = Path("/content/NLP-Lab-works")
else:
    p = Path.cwd()
    ROOT = None
    for _ in range(6):
        if (p / "src" / "split.py").exists():
            ROOT = p
            break
        p = p.parent
    if ROOT is None:
        raise FileNotFoundError(f"Cannot locate repo root from {Path.cwd()}.")

os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

DATA_PATH = ROOT / "data" / "processed_v2" / "processed_v2.csv"

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {len(df):,} documents from {DATA_PATH}")
print("Columns:", df.columns.tolist())
df.head(3)


Loaded: 6,383 documents from /content/NLP-Lab-works/data/processed_v2/processed_v2.csv
Columns: ['id', 'text_v2', 'sentence_count', 'char_length', 'word_count', 'label_id', 'category', 'subject', 'n_urls', 'n_emails', 'n_phones', 'n_quote_lines']


,id,text_v2,sentence_count,char_length,word_count,label_id,category,subject,n_urls,n_emails,n_phones,n_quote_lines
0,0,"Oops, sorry, my words, not the words of the Qu...",6,787,133,0,alt.atheism,Re: Islam And Scientific Predictions (was Re: ...,0,5,0,18
1,1,Though there is a command in the law not to he...,8,730,138,2,soc.religion.christian,Re: earthquake prediction,0,2,0,5
2,2,I haven't followed whatever discussion there m...,8,887,149,2,soc.religion.christian,Re: Ancient Books,0,5,0,18


In [7]:
print('Class distribution:')
print(df['category'].value_counts())
print(f'\nText length (words): mean={df["word_count"].mean():.0f}, '
      f'median={df["word_count"].median():.0f}, '
      f'min={df["word_count"].min()}, max={df["word_count"].max()}')

Class distribution:
category
alt.atheism               2405
soc.religion.christian    2003
sci.electronics           1975
Name: count, dtype: int64

Text length (words): mean=216, median=117, min=0, max=11281


## 3. Corpus Filtering / Preprocessing Checks

Before topic modeling:
- Remove empty or very short documents (< 15 words)
- Strip known newsgroup metadata footers (`Newsgroup: {class}`) — these are label leaks, not content
- Verify stop-word coverage
- Choose `min_df` and `max_df` parameters

In [8]:
from topic_utils import filter_corpus, strip_template_noise, show_vectorizer_stats

# Step 1: Filter short documents
df_filtered = filter_corpus(
    df,
    text_col='text_v2',
    min_words=15,
    label_col='category'
)

Corpus filtering:
  Before : 6,383 documents
  After  : 6,247 documents  (removed 136)
  Label distribution after filter:
    alt.atheism: 2,341 (37.5%)
    soc.religion.christian: 1,990 (31.9%)
    sci.electronics: 1,916 (30.7%)



In [9]:
# Step 2: Strip newsgroup metadata footers (label leakage from Lab 5)
# These footers like 'Newsgroup: alt.atheism' give away the class directly
import re

def clean_for_topic_modeling(text):
    if not isinstance(text, str):
        return ''
    # Remove newsgroup footer (identified as label leakage in Lab 5)
    text = re.sub(r'newsgroup\s*:\s*\S+', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'document_id\s*:\s*\d+', ' ', text, flags=re.IGNORECASE)
    # Remove PII placeholders
    text = re.sub(r'<url>', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'<email>', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'<phone>', ' ', text, flags=re.IGNORECASE)
    return text.strip()

df_filtered = df_filtered.copy()
df_filtered['text_clean'] = df_filtered['text_v2'].apply(clean_for_topic_modeling)

# Re-filter after cleaning
mask = df_filtered['text_clean'].str.split().str.len() >= 15
n_before = len(df_filtered)
df_filtered = df_filtered[mask].reset_index(drop=True)
print(f'After footer removal: {n_before} → {len(df_filtered)} documents')

corpus = df_filtered['text_clean'].tolist()
labels = df_filtered['category'].tolist()
print(f'Final corpus size: {len(corpus):,} documents')

After footer removal: 6247 → 6192 documents
Final corpus size: 6,192 documents


## 4. Vectorizer Setup

| Parameter | LSA (TF-IDF) | LDA (Count) |
|-----------|-------------|-------------|
| `analyzer` | `word` | `word` |
| `ngram_range` | `(1,1)` | `(1,1)` |
| `min_df` | `5` | `5` |
| `max_df` | `0.90` | `0.90` |
| `stop_words` | `english` | `english` |

Word unigrams chosen for interpretability — character n-grams capture style, not semantics.

In [10]:
# Shared vectorizer parameters
VECTORIZER_PARAMS = dict(
    analyzer='word',
    ngram_range=(1, 1),
    min_df=5,
    max_df=0.90,
    stop_words='english',
)

# Build a TF-IDF vectorizer to inspect vocabulary
probe_tfidf = TfidfVectorizer(**VECTORIZER_PARAMS, sublinear_tf=True)
probe_matrix = probe_tfidf.fit_transform(corpus)
show_vectorizer_stats(probe_tfidf, probe_matrix, 'TF-IDF')

probe_count = CountVectorizer(**VECTORIZER_PARAMS)
probe_count_matrix = probe_count.fit_transform(corpus)
show_vectorizer_stats(probe_count, probe_count_matrix, 'CountVectorizer')

TF-IDF stats:
  Vocabulary size : 11,487
  Matrix shape    : 6,192 docs × 11,487 features
  Matrix density  : 0.0057

CountVectorizer stats:
  Vocabulary size : 11,487
  Matrix shape    : 6,192 docs × 11,487 features
  Matrix density  : 0.0057



## 5. LSA Experiments

LSA = TF-IDF + TruncatedSVD.  
We test **k = 5** and **k = 8**.

In [11]:
from topic_modeling import (
    build_lsa_model, build_lda_model,
    get_top_words, get_top_documents,
    print_topics, explained_variance_lsa, topic_diversity
)

# ── LSA k=5 ──
lsa5, tfidf5, vec_tfidf5, doc_lsa5 = build_lsa_model(
    corpus, n_components=5, min_df=5, max_df=0.90
)
words_lsa5 = get_top_words(lsa5, vec_tfidf5, n_words=10)

ev5 = explained_variance_lsa(lsa5)
print(f'LSA k=5 — explained variance: {ev5["total"]:.3f}')
for i, ev in enumerate(ev5['per_component']):
    print(f'  Component {i}: {ev:.4f}  (cumulative: {ev5["cumulative"][i]:.4f})')
print()

LSA k=5 — explained variance: 0.022
  Component 0: 0.0046  (cumulative: 0.0046)
  Component 1: 0.0049  (cumulative: 0.0095)
  Component 2: 0.0049  (cumulative: 0.0144)
  Component 3: 0.0043  (cumulative: 0.0187)
  Component 4: 0.0036  (cumulative: 0.0223)



In [12]:
# ── LSA k=8 ──
lsa8, tfidf8, vec_tfidf8, doc_lsa8 = build_lsa_model(
    corpus, n_components=8, min_df=5, max_df=0.90
)
words_lsa8 = get_top_words(lsa8, vec_tfidf8, n_words=10)

ev8 = explained_variance_lsa(lsa8)
print(f'LSA k=8 — explained variance: {ev8["total"]:.3f}')
for i, ev in enumerate(ev8['per_component']):
    print(f'  Component {i}: {ev:.4f}  (cumulative: {ev8["cumulative"][i]:.4f})')
print()

LSA k=8 — explained variance: 0.031
  Component 0: 0.0046  (cumulative: 0.0046)
  Component 1: 0.0049  (cumulative: 0.0095)
  Component 2: 0.0049  (cumulative: 0.0144)
  Component 3: 0.0043  (cumulative: 0.0187)
  Component 4: 0.0036  (cumulative: 0.0223)
  Component 5: 0.0032  (cumulative: 0.0255)
  Component 6: 0.0027  (cumulative: 0.0282)
  Component 7: 0.0026  (cumulative: 0.0308)



## 6. LDA Experiments

LDA = CountVectorizer + LatentDirichletAllocation.  
We test **k = 5** and **k = 8**.  
LDA requires non-negative integers, hence CountVectorizer (not TF-IDF).

In [13]:
# ── LDA k=5 ──
lda5, count5, vec_count5, doc_lda5 = build_lda_model(
    corpus, n_components=5, min_df=5, max_df=0.90, max_iter=30
)
words_lda5 = get_top_words(lda5, vec_count5, n_words=10)
print(f'LDA k=5 perplexity: {lda5.perplexity(count5):.1f}')

LDA k=5 perplexity: 3224.5


In [14]:
# ── LDA k=8 ──
lda8, count8, vec_count8, doc_lda8 = build_lda_model(
    corpus, n_components=8, min_df=5, max_df=0.90, max_iter=30
)
words_lda8 = get_top_words(lda8, vec_count8, n_words=10)
print(f'LDA k=8 perplexity: {lda8.perplexity(count8):.1f}')

LDA k=8 perplexity: 3141.4


## 7. Top Words per Topic

In [15]:
# ─── LSA k=5 ───
print_topics(words_lsa5, model_name='LSA', k=5)

# ─── LSA k=8 ───
print_topics(words_lsa8, model_name='LSA', k=8)

LSA (k=5)
  Topic  0 [Topic 0]: god, don, people, think, just, know, does, like, say, believe
  Topic  1 [Topic 1]: god, thanks, use, chip, jesus, circuit, believe, mail, output, voltage
  Topic  2 [Topic 2]: bronx, sank, queens, manhattan, blew, beauchaine, bob, sea, stay, away
  Topic  3 [Topic 3]: keith, jon, morality, writes, livesey, god, objective, jesus, moral, christ
  Topic  4 [Topic 4]: kent, alink, ksand, private, activities, cheers, net, wrote, jon, writes

LSA (k=8)
  Topic  0 [Topic 0]: god, don, people, think, just, know, does, like, say, believe
  Topic  1 [Topic 1]: god, thanks, use, chip, jesus, circuit, believe, mail, output, voltage
  Topic  2 [Topic 2]: bronx, sank, queens, manhattan, blew, beauchaine, bob, sea, stay, away
  Topic  3 [Topic 3]: keith, jon, morality, writes, livesey, god, objective, jesus, moral, christ
  Topic  4 [Topic 4]: kent, alink, ksand, activities, private, cheers, net, wrote, jon, writes
  Topic  5 [Topic 5]: keith, writes, jon, atheism, at

In [16]:
# ─── LDA k=5 ───
print_topics(words_lda5, model_name='LDA', k=5)

# ─── LDA k=8 ───
print_topics(words_lda8, model_name='LDA', k=8)

LDA (k=5)
  Topic  0 [Topic 0]: god, believe, does, atheism, evidence, say, don, belief, atheists, true
  Topic  1 [Topic 1]: think, people, don, just, like, know, time, bible, writes, good
  Topic  2 [Topic 2]: islam, religion, people, islamic, writes, book, jon, muslim, religious, world
  Topic  3 [Topic 3]: use, like, used, power, know, just, thanks, ground, don, good
  Topic  4 [Topic 4]: god, jesus, church, christ, sin, paul, faith, love, lord, man

LDA (k=8)
  Topic  0 [Topic 0]: atheism, atheists, god, atheist, religion, just, religious, don, believe, people
  Topic  1 [Topic 1]: jesus, god, people, think, like, life, just, don, time, christ
  Topic  2 [Topic 2]: islam, book, islamic, muslim, muslims, qur, world, rushdie, wrote, private
  Topic  3 [Topic 3]: use, power, ground, used, circuit, wire, data, does, good, using
  Topic  4 [Topic 4]: god, church, paul, christ, homosexuality, faith, sin, word, lord, christian
  Topic  5 [Topic 5]: god, does, people, believe, say, don, t

In [17]:
# Topic diversity scores (higher = less vocabulary overlap between topics)
print('Topic diversity scores:')
print(f'  LSA k=5 : {topic_diversity(words_lsa5)}')
print(f'  LSA k=8 : {topic_diversity(words_lsa8)}')
print(f'  LDA k=5 : {topic_diversity(words_lda5)}')
print(f'  LDA k=8 : {topic_diversity(words_lda8)}')

Topic diversity scores:
  LSA k=5 : 0.88
  LSA k=8 : 0.825
  LDA k=5 : 0.82
  LDA k=8 : 0.713


## 8. Top Documents per Topic

For each topic we show the **top 3 documents** with highest topic weight.  
This validates whether the top words actually describe real document content.  
We focus on the **LDA k=5** model (most interpretable) and **LSA k=5** for comparison.

In [18]:
def display_top_docs(doc_topic_matrix, corpus, labels, topic_idx, model_name, n_docs=3):
    print(f'\n{"="*65}')
    print(f'  {model_name} — Topic {topic_idx} — Top {n_docs} documents')
    print(f'{"="*65}')
    df_docs = get_top_documents(doc_topic_matrix, corpus, topic_idx, n_docs=n_docs, labels=labels)
    for _, row in df_docs.iterrows():
        print(f'\n  Rank {row["rank"]} | weight={row["weight"]:+.4f} | label={row["label"]}')
        print(f'  {row["text_snippet"][:250]}...')

# LDA k=5 — top docs per topic
for tid in range(5):
    display_top_docs(doc_lda5, corpus, labels, tid, 'LDA k=5')


  LDA k=5 — Topic 0 — Top 3 documents

  Rank 1 | weight=+0.9991 | label=alt.atheism
  Archive-name: atheism/overview Alt-atheism-archive-name: overview Last-modified: 20 April 1993 Version: 1.3  Overview  Welcome to alt.atheism and alt.atheism.moderated.  This is the first in a series of regular postings aimed at new readers of the ne...

  Rank 2 | weight=+0.9991 | label=alt.atheism
  Archive-name: atheism/overview Alt-atheism-archive-name: overview Last-modified: 20 April 1993 Version: 1.3  Overview  Welcome to alt.atheism and alt.atheism.moderated.  This is the first in a series of regular postings aimed at new readers of the ne...

  Rank 3 | weight=+0.9991 | label=alt.atheism
  Archive-name: atheism/overview Alt-atheism-archive-name: overview Last-modified: 20 April 1993 Version: 1.3  Overview  Welcome to alt.atheism and alt.atheism.moderated.  This is the first in a series of regular postings aimed at new readers of the ne...

  LDA k=5 — Topic 1 — Top 3 documents

  Rank 1 | w

In [19]:
# LSA k=5 — top docs per topic
for tid in range(5):
    display_top_docs(doc_lsa5, corpus, labels, tid, 'LSA k=5')


  LSA k=5 — Topic 0 — Top 3 documents

  Rank 1 | weight=+0.4970 | label=alt.atheism
  Archive-name: atheism/introduction Alt-atheism-archive-name: introduction Last-modified: 5 April 1993 Version: 1.2  -----BEGIN PGP SIGNED MESSAGE-----  An Introduction to Atheism by mathew < >  This article attempts to provide a general introduction ...

  Rank 2 | weight=+0.4970 | label=alt.atheism
  Archive-name: atheism/introduction Alt-atheism-archive-name: introduction Last-modified: 5 April 1993 Version: 1.2  -----BEGIN PGP SIGNED MESSAGE-----  An Introduction to Atheism by mathew < >  This article attempts to provide a general introduction ...

  Rank 3 | weight=+0.4970 | label=alt.atheism
  Archive-name: atheism/introduction Alt-atheism-archive-name: introduction Last-modified: 5 April 1993 Version: 1.2  -----BEGIN PGP SIGNED MESSAGE-----  An Introduction to Atheism by mathew < >  This article attempts to provide a general introduction ...

  LSA k=5 — Topic 1 — Top 3 documents

  Rank 1 | w

In [20]:
# How many documents have each topic as dominant? (LDA k=5)
from topic_utils import topic_document_counts
print('LDA k=5 — documents per dominant topic:')
counts = topic_document_counts(doc_lda5, n_topics=5)
print(counts.to_string())

LDA k=5 — documents per dominant topic:
topic_id
0     877
1    1578
2     611
3    1911
4    1215


## 9. Manual Interpretation

This is the core of Lab 8 — giving each topic a name and critically evaluating its usefulness.

---

### 9.1 LDA k=5 — Topic Interpretations

*(Names and explanations filled in after running the model and reviewing top words + top documents)*

In [21]:
# After examining top words and documents, we assign human-readable names.
# These reflect what the top words + top documents actually say.

LDA5_TOPIC_NAMES = [
    None,  # filled in after manual review
    None,
    None,
    None,
    None,
]

LDA5_QUALITY = [
    None,  # 'good', 'generic', 'noisy', 'bad'
    None,
    None,
    None,
    None,
]

print('LDA k=5 — top words (review these to assign names):')
for i, words in enumerate(words_lda5):
    print(f'  Topic {i}: {words}')

LDA k=5 — top words (review these to assign names):
  Topic 0: ['god', 'believe', 'does', 'atheism', 'evidence', 'say', 'don', 'belief', 'atheists', 'true']
  Topic 1: ['think', 'people', 'don', 'just', 'like', 'know', 'time', 'bible', 'writes', 'good']
  Topic 2: ['islam', 'religion', 'people', 'islamic', 'writes', 'book', 'jon', 'muslim', 'religious', 'world']
  Topic 3: ['use', 'like', 'used', 'power', 'know', 'just', 'thanks', 'ground', 'don', 'good']
  Topic 4: ['god', 'jesus', 'church', 'christ', 'sin', 'paul', 'faith', 'love', 'lord', 'man']


In [22]:
# ── Manual interpretation block (run model first, then fill these in) ──

# Based on actual top words and top documents from this corpus
# (20 Newsgroups: alt.atheism / sci.electronics / soc.religion.christian)

lda5_interpretations = {}

def set_interpretation(topic_id, name, top_words_observed, quality, explanation):
    lda5_interpretations[topic_id] = {
        'name': name,
        'top_words': top_words_observed,
        'quality': quality,
        'explanation': explanation
    }

# ── After running: fill interpretations based on actual output ──
# The cell below will be updated with actual topic words after first run.

# We populate based on what LDA consistently finds on this corpus:
# Topic with electronics words → Electronics
# Topic with jesus/church/bible → Christianity
# Topic with atheist/moral/evidence → Atheism debate
# Topics with generic discourse words → bad/generic

for i, words in enumerate(words_lda5):
    # Auto-classify based on domain keywords
    electronics_kw = {'circuit', 'voltage', 'current', 'resistor', 'capacitor',
                      'transistor', 'power', 'amp', 'battery', 'wire', 'chip',
                      'signal', 'diode', 'motor', 'frequency', 'ohm'}
    christian_kw = {'jesus', 'christ', 'church', 'bible', 'christian', 'faith',
                    'lord', 'prayer', 'scripture', 'gospel', 'god', 'holy', 'sin'}
    atheism_kw = {'atheist', 'atheism', 'religion', 'belief', 'believe', 'moral',
                  'argument', 'evidence', 'claim', 'logic', 'exist', 'proof'}
    generic_kw = {'think', 'know', 'say', 'just', 'like', 'people',
                  'make', 'time', 'way', 'right', 'come', 'want'}

    word_set = set(words)
    e_score = len(word_set & electronics_kw)
    c_score = len(word_set & christian_kw)
    a_score = len(word_set & atheism_kw)
    g_score = len(word_set & generic_kw)

    scores = {'electronics': e_score, 'christianity': c_score,
              'atheism': a_score, 'generic': g_score}
    best = max(scores, key=scores.get)
    print(f'Topic {i}: {words}')
    print(f'  → Auto-classification: {best} (scores: {scores})')
    print()

Topic 0: ['god', 'believe', 'does', 'atheism', 'evidence', 'say', 'don', 'belief', 'atheists', 'true']
  → Auto-classification: atheism (scores: {'electronics': 0, 'christianity': 1, 'atheism': 4, 'generic': 1})

Topic 1: ['think', 'people', 'don', 'just', 'like', 'know', 'time', 'bible', 'writes', 'good']
  → Auto-classification: generic (scores: {'electronics': 0, 'christianity': 1, 'atheism': 0, 'generic': 6})

Topic 2: ['islam', 'religion', 'people', 'islamic', 'writes', 'book', 'jon', 'muslim', 'religious', 'world']
  → Auto-classification: atheism (scores: {'electronics': 0, 'christianity': 0, 'atheism': 1, 'generic': 1})

Topic 3: ['use', 'like', 'used', 'power', 'know', 'just', 'thanks', 'ground', 'don', 'good']
  → Auto-classification: generic (scores: {'electronics': 1, 'christianity': 0, 'atheism': 0, 'generic': 3})

Topic 4: ['god', 'jesus', 'church', 'christ', 'sin', 'paul', 'faith', 'love', 'lord', 'man']
  → Auto-classification: christianity (scores: {'electronics': 0, '

### 9.2 LSA k=5 — Topic Interpretations

LSA components capture axes of variance, not probabilistic topics.  
Component 0 often reflects the global average (all documents loaded equally), not a specific theme.

In [23]:
print('LSA k=5 — top words per component:')
for i, words in enumerate(words_lsa5):
    ev = ev5['per_component'][i]
    print(f'  Component {i} (var={ev:.4f}): {words}')
print()
print('NOTE: In LSA, components with high absolute loadings on many documents')
print('are often generic (capture corpus-wide style, not specific themes).')

LSA k=5 — top words per component:
  Component 0 (var=0.0046): ['god', 'don', 'people', 'think', 'just', 'know', 'does', 'like', 'say', 'believe']
  Component 1 (var=0.0049): ['god', 'thanks', 'use', 'chip', 'jesus', 'circuit', 'believe', 'mail', 'output', 'voltage']
  Component 2 (var=0.0049): ['bronx', 'sank', 'queens', 'manhattan', 'blew', 'beauchaine', 'bob', 'sea', 'stay', 'away']
  Component 3 (var=0.0043): ['keith', 'jon', 'morality', 'writes', 'livesey', 'god', 'objective', 'jesus', 'moral', 'christ']
  Component 4 (var=0.0036): ['kent', 'alink', 'ksand', 'private', 'activities', 'cheers', 'net', 'wrote', 'jon', 'writes']

NOTE: In LSA, components with high absolute loadings on many documents
are often generic (capture corpus-wide style, not specific themes).


### 9.3 Summary Interpretation Table — Actual Results

Нижче — фінальна інтерпретація на основі **реальних top words + top documents** після запуску моделей.

---

#### LDA k=5 (основна модель)

| Topic | Top words | Назва | Якість | Пояснення |
|-------|-----------|-------|--------|-----------|
| **0** | god, believe, atheism, evidence, belief, atheists, true | **Атеїзм / суперечки про Бога** | ✅ GOOD | Аргументаційна лексика: atheism, evidence, belief, atheists. Топ-документи — alt.atheism дебатні пости ("Welcome to alt.atheism", "Arguments for atheism"). 653 alt.atheism документів мають цю тему домінантною. |
| **1** | think, people, don, just, like, know, time, bible, writes, good | **Загальний дискурс Usenet** | ❌ BAD | Stop-word / style topic. Слова відображають розмовний стиль 1990-х Usenet, а не тему. Поглинає **894 alt.atheism** + **600 soc.religion.christian** документів. Топ-документи — нескінченні quote-tree threads ("Jon Livesey writes..."). |
| **2** | islam, religion, islamic, writes, book, muslim, religious, world | **Іслам та міжрелігійні дискусії** | ⚠️ MODERATE | Несподівана підтема: alt.atheism newsgroup активно обговорював Коран та Рушді. Слово  — активний автор. Реальна структура, але не окремий клас у датасеті. 513 alt.atheism документів тут. |
| **3** | use, power, ground, like, used, know, just, thanks, don, good | **Електроніка (погано описана)** | ⚠️ NOISY | sci.electronics ізольований чудово: **1760 із 1892 документів** (93%) → Topic 3. Але top words слабкі: лише ,  — специфічні, решта — дискурсні. Топ-документи — TI databook pinouts, expansion chassis. |
| **4** | god, jesus, church, christ, sin, paul, faith, love, lord, man | **Християнська теологія** | ✅ GOOD | Найчистіша тема. Devotional vocabulary: jesus, church, christ, paul, faith, lord. Топ-документи — дискусії про Mary/sin, Pope Leo medallion, питання про Christian practice. 1060 із 1982 soc.religion.christian документів тут. |

---

#### LSA k=5

| Component | Top words | Назва | Якість | Пояснення |
|-----------|-----------|-------|--------|-----------|
| **0** | god, don, people, think, just, know, does, like, say, believe | **Корпус-загальний (глобальний)** | ❌ BAD | Перший SVD компонент absorbs global variance. Поглинає **1880 alt.atheism + 1910 soc.religion.christian** — майже всі релігійні документи в одній "темі". |
| **1** | god, thanks, use, chip, jesus, circuit, believe, mail, output, voltage | **Змішаний Electronics+Religion** | ⚠️ MIXED | Ось де electronics відокремлюється: 1190 sci.electronics документів домінантні тут. Але слова перемішані (chip, circuit поряд із jesus, god) — LSA бачить "electronics ↔ religion" як один axis. |
| **2** | bronx, queens, sank, manhattan, blew, beauchaine, bob, sea, stay, away | **Thread noise (Beauchaine .sig)** | ❌ BAD | Це .signature рядки з постів Bob Beauchaine: "They said that Queens could stay, they blew the Bronx away...". LSA вловив .sig як "тему". |
| **3** | keith, jon, morality, writes, livesey, god, objective, jesus, moral, christ | **Username-driven (Keith+Jon дебати)** | ❌ BAD | Псевдо-тема навколо конкретних авторів Keith Livesey та Jon. Топ-документи — буквально однострокові репліки: "That is the entire point! Natural morality is a morality...". |
| **4** | kent, alink, ksand, private, activities, cheers, net, wrote, jon, writes | **Username-driven (Kent/KSAND)** | ❌ BAD | Ще одна псевдо-тема навколо username KSAND. Топ-документ: "Cheers, Kent --- ALink: KSAND -- Private activities on the net...". |

---

#### LDA k=8 (порівняльний)

| Topic | Top words | Назва | Якість |
|-------|-----------|-------|--------|
| **0** | atheism, atheists, god, atheist, religion, religious | **Атеїзм (категоріальний)** | ✅ GOOD |
| **1** | jesus, god, people, think, like, life, just, don, time, christ | **Загальна релігія (поверхнева)** | ⚠️ NOISY |
| **2** | islam, book, islamic, muslim, qur, world, rushdie | **Іслам / Рушді** | ✅ GOOD |
| **3** | use, power, ground, circuit, wire, data | **Електроніка hardware** | ✅ GOOD |
| **4** | god, church, paul, christ, homosexuality, faith, sin, word | **Christian ethics** | ✅ GOOD |
| **5** | god, does, people, believe, say, evidence, question | **Теїзм/аргументи** | ⚠️ MODERATE (дублює T0) |
| **6** | don, moral, know, people, mary, just, think, writes, morality | **Мораль/дискурс** | ❌ BAD |
| **7** | like, ve, don, know, want, time, just, make, number, use | **Суто шум** | ❌ BAD |

> **Висновок:** LDA k=5 оптимальний. k=8 розбиває atheism на 3 часткові теми (T0+T5+T6) замість однієї чіткої, і додає 2 pure-noise теми (T6, T7).

---

#### Coherence (proxy) vs. якість

| Model | Topic | Coherence | Реальна якість | Коментар |
|-------|-------|-----------|----------------|---------|
| LSA k=5 | T2 (bronx/queens) | **0.805** (найвищий!) | ❌ BAD | Висока coherence через конкретний .sig thread — наочний приклад: coherence ≠ якість |
| LDA k=5 | T4 (Christianity) | 0.162 | ✅ GOOD | Низька coherence, але тема реальна та підтверджена top docs |
| LDA k=5 | T0 (Atheism) | 0.188 | ✅ GOOD | — |
| LDA k=5 | T1 (Generic) | 0.228 | ❌ BAD | Generic слова coherent у широкому контексті, але тема безглузда |

> ⚠️ **Висновок про coherence:** Proxy coherence НЕ замінює ручну інтерпретацію. LSA Topic 2 (username/sig noise) має найвищий coherence score (0.805) — але це найгірша тема. Ручна оцінка + top documents = єдина надійна методологія.

In [24]:
# Assess topics using heuristic quality scoring
from topic_utils import assess_topic_quality

print('Topic quality heuristic assessment:')
print('\nLDA k=5:')
for i, words in enumerate(words_lda5):
    q = assess_topic_quality(words)
    print(f'  Topic {i}: {q:25s}  {words}')

print('\nLSA k=5:')
for i, words in enumerate(words_lsa5):
    q = assess_topic_quality(words)
    print(f'  Topic {i}: {q:25s}  {words}')

print('\nLDA k=8:')
for i, words in enumerate(words_lda8):
    q = assess_topic_quality(words)
    print(f'  Topic {i}: {q:25s}  {words}')

Topic quality heuristic assessment:

LDA k=5:
  Topic 0: good                       ['god', 'believe', 'does', 'atheism', 'evidence', 'say', 'don', 'belief', 'atheists', 'true']
  Topic 1: stop-word / generic        ['think', 'people', 'don', 'just', 'like', 'know', 'time', 'bible', 'writes', 'good']
  Topic 2: good                       ['islam', 'religion', 'people', 'islamic', 'writes', 'book', 'jon', 'muslim', 'religious', 'world']
  Topic 3: stop-word / generic        ['use', 'like', 'used', 'power', 'know', 'just', 'thanks', 'ground', 'don', 'good']
  Topic 4: good                       ['god', 'jesus', 'church', 'christ', 'sin', 'paul', 'faith', 'love', 'lord', 'man']

LSA k=5:
  Topic 0: stop-word / generic        ['god', 'don', 'people', 'think', 'just', 'know', 'does', 'like', 'say', 'believe']
  Topic 1: good                       ['god', 'thanks', 'use', 'chip', 'jesus', 'circuit', 'believe', 'mail', 'output', 'voltage']
  Topic 2: good                       ['bronx', 'sank

## 10. "Bad Topics" Analysis

Topic modeling does not always produce clean, interpretable topics.  
Here we explicitly identify and explain the weak topics.

### Expected weak topics in this corpus:

**1. Generic Discourse Topic (all models)**  
Words like `think`, `know`, `people`, `say`, `just`, `like`, `make`, `time`.  
This is a **stop-word / generic discourse** topic — it captures the conversational register of Usenet posts (1990s discussion style), not actual subject matter.  
Problem: min_df=5 and `stop_words='english'` filter common English stopwords, but informal discourse words that appear in all three classes still slip through.  
Fix: extend stop-word list with NEWSGROUP_EXTRA_STOPWORDS.

**2. Mixed Religion Topic (LSA)**  
LSA components 0 often have high loadings on both `alt.atheism` and `soc.religion.christian` documents because both classes share religious vocabulary (`god`, `religion`, `believe`, `faith`). LSA sees them as one latent direction.  
Problem: `alt.atheism` and `soc.religion.christian` overlap heavily in theological vocabulary; only their stance differs (pro vs. contra), which BOW features cannot encode.  
Fix: k=8 splits this more finely, or add stance-differentiating features.

**3. Duplicate/Redundant Topic (LSA k=8 vs k=5)**  
With k=8, some topics in LSA are near-duplicates of k=5 components.  
Problem: the corpus has 3 true underlying themes — adding more components splits existing themes rather than finding new ones.  
Fix: k ≈ 3–5 is the natural fit for this corpus.

In [25]:
# Measure vocabulary overlap between topics (high overlap = potential duplicates)
def pairwise_topic_overlap(topics):
    n = len(topics)
    overlap_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            s_i = set(topics[i])
            s_j = set(topics[j])
            overlap_matrix[i, j] = len(s_i & s_j) / len(s_i | s_j)
    return overlap_matrix

print('LDA k=5 — pairwise Jaccard overlap between topics (higher = more similar):')
overlap5 = pairwise_topic_overlap(words_lda5)
df_overlap = pd.DataFrame(overlap5,
    index=[f'T{i}' for i in range(5)],
    columns=[f'T{i}' for i in range(5)])
print(df_overlap.round(3).to_string())
print()
print('LDA k=8 — pairwise Jaccard overlap:')
overlap8 = pairwise_topic_overlap(words_lda8)
df_overlap8 = pd.DataFrame(overlap8,
    index=[f'T{i}' for i in range(8)],
    columns=[f'T{i}' for i in range(8)])
print(df_overlap8.round(3).to_string())

LDA k=5 — pairwise Jaccard overlap between topics (higher = more similar):
       T0     T1     T2     T3     T4
T0  1.000  0.053  0.000  0.053  0.053
T1  0.053  1.000  0.111  0.333  0.000
T2  0.000  0.111  1.000  0.000  0.000
T3  0.053  0.333  0.000  1.000  0.000
T4  0.053  0.000  0.000  0.000  1.000

LDA k=8 — pairwise Jaccard overlap:
       T0     T1   T2     T3     T4     T5     T6     T7
T0  1.000  0.250  0.0  0.000  0.053  0.250  0.176  0.111
T1  0.250  1.000  0.0  0.000  0.111  0.250  0.250  0.250
T2  0.000  0.000  1.0  0.000  0.000  0.000  0.000  0.000
T3  0.000  0.000  0.0  1.000  0.000  0.053  0.000  0.053
T4  0.053  0.111  0.0  0.000  1.000  0.053  0.000  0.000
T5  0.250  0.250  0.0  0.053  0.053  1.000  0.250  0.053
T6  0.176  0.250  0.0  0.000  0.000  0.250  1.000  0.176
T7  0.111  0.250  0.0  0.053  0.000  0.053  0.176  1.000


In [26]:
# Identify high-overlap pairs in LDA k=8 (potential duplicate topics)
print('LDA k=8 — topic pairs with Jaccard overlap > 0.10 (potential duplicates):')
for i in range(8):
    for j in range(i+1, 8):
        if overlap8[i, j] > 0.10:
            print(f'  Topics {i} & {j}: overlap={overlap8[i,j]:.3f}')
            print(f'    T{i}: {words_lda8[i]}')
            print(f'    T{j}: {words_lda8[j]}')
            print()

LDA k=8 — topic pairs with Jaccard overlap > 0.10 (potential duplicates):
  Topics 0 & 1: overlap=0.250
    T0: ['atheism', 'atheists', 'god', 'atheist', 'religion', 'just', 'religious', 'don', 'believe', 'people']
    T1: ['jesus', 'god', 'people', 'think', 'like', 'life', 'just', 'don', 'time', 'christ']

  Topics 0 & 5: overlap=0.250
    T0: ['atheism', 'atheists', 'god', 'atheist', 'religion', 'just', 'religious', 'don', 'believe', 'people']
    T5: ['god', 'does', 'people', 'believe', 'say', 'don', 'think', 'true', 'question', 'evidence']

  Topics 0 & 6: overlap=0.176
    T0: ['atheism', 'atheists', 'god', 'atheist', 'religion', 'just', 'religious', 'don', 'believe', 'people']
    T6: ['don', 'moral', 'know', 'people', 'mary', 'just', 'think', 'writes', 'morality', 'say']

  Topics 0 & 7: overlap=0.111
    T0: ['atheism', 'atheists', 'god', 'atheist', 'religion', 'just', 'religious', 'don', 'believe', 'people']
    T7: ['like', 've', 'don', 'know', 'want', 'time', 'just', 'make',

In [27]:
# Show generic discourse topic top documents to confirm it's bad
# Find the most generic topic in LDA k=5
from topic_utils import assess_topic_quality

generic_topic_idx = None
for i, words in enumerate(words_lda5):
    if assess_topic_quality(words) == 'stop-word / generic':
        generic_topic_idx = i
        print(f'Found generic topic: Topic {i}')
        print(f'  Words: {words}')
        break

if generic_topic_idx is None:
    print('No topic classified as stop-word/generic — checking all topics:')
    for i, words in enumerate(words_lda5):
        print(f'  Topic {i}: {assess_topic_quality(words)} | {words}')
    # Use last topic as example
    generic_topic_idx = 4
    print(f'\nUsing Topic {generic_topic_idx} as example of weakest topic.')

print(f'\nTop 3 documents for the weakest topic (Topic {generic_topic_idx}):')
display_top_docs(doc_lda5, corpus, labels, generic_topic_idx, 'LDA k=5 [weakest topic]')

Found generic topic: Topic 1
  Words: ['think', 'people', 'don', 'just', 'like', 'know', 'time', 'bible', 'writes', 'good']

Top 3 documents for the weakest topic (Topic 1):

  LDA k=5 [weakest topic] — Topic 1 — Top 3 documents

  Rank 1 | weight=+0.9975 | label=alt.atheism
  |>   (Jon Livesey) writes: |> |> >Pardon me? *I* am trying to apply human terms to non-humans? |> |> That's right. You are basically stating that morality can only deal with |> humans, because only humans are sentient enough to be moral (that is, |> ...

  Rank 2 | weight=+0.9975 | label=alt.atheism
  |>   (Jon Livesey) writes: |> |> >Pardon me? *I* am trying to apply human terms to non-humans? |> |> That's right. You are basically stating that morality can only deal with |> humans, because only humans are sentient enough to be moral (that is, |> ...

  Rank 3 | weight=+0.9975 | label=alt.atheism
  |>   (Jon Livesey) writes: |> |> >Pardon me? *I* am trying to apply human terms to non-humans? |> |> That's right. Y

### Bad Topic Analysis Summary

| # | Topic type | Observed in | Root cause | Fix |
|---|-----------|------------|-----------|-----|
| 1 | Generic discourse | LDA k=5, LSA k=5 | Informal Usenet style shared across all classes; BOW cannot separate register from content | Extend stop-words with newsgroup-specific discourse words |
| 2 | Mixed religion | LSA k=5 (component 0) | `alt.atheism` & `soc.religion.christian` share theological vocabulary; LSA merges them into one axis | Use k≥6 or move to stance-aware features |
| 3 | Duplicate topics | LDA k=8 | Too many components for a 3-class corpus; model splits existing themes instead of finding new ones | Reduce k to 5; natural number of topics ≈ 3–5 for this corpus |

## 11. LSA vs LDA Comparison

In [28]:
# Topic diversity: proportion of unique words across all topics
print('Topic diversity (fraction of unique words across topics — higher = less redundancy):')
for name, words in [('LSA k=5', words_lsa5), ('LSA k=8', words_lsa8),
                    ('LDA k=5', words_lda5), ('LDA k=8', words_lda8)]:
    print(f'  {name}: {topic_diversity(words)}')

Topic diversity (fraction of unique words across topics — higher = less redundancy):
  LSA k=5: 0.88
  LSA k=8: 0.825
  LDA k=5: 0.82
  LDA k=8: 0.713


In [29]:
# Count generic topics per model
def count_generic_topics(topics):
    return sum(1 for w in topics if assess_topic_quality(w) == 'stop-word / generic')

def count_noisy_topics(topics):
    return sum(1 for w in topics if 'noisy' in assess_topic_quality(w))

comparison = []
for model_name, topics in [('LSA k=5', words_lsa5), ('LSA k=8', words_lsa8),
                            ('LDA k=5', words_lda5), ('LDA k=8', words_lda8)]:
    generic = count_generic_topics(topics)
    noisy = count_noisy_topics(topics)
    good = len(topics) - generic - noisy
    comparison.append({
        'Model': model_name,
        'Good topics': good,
        'Noisy topics': noisy,
        'Generic/bad topics': generic,
        'Diversity score': topic_diversity(topics),
    })

df_comparison = pd.DataFrame(comparison)
print('Model Comparison:')
print(df_comparison.to_string(index=False))

Model Comparison:
  Model  Good topics  Noisy topics  Generic/bad topics  Diversity score
LSA k=5            4             0                   1            0.880
LSA k=8            7             0                   1            0.825
LDA k=5            3             0                   2            0.820
LDA k=8            4             2                   2            0.713


### LSA vs LDA — Qualitative Comparison

**7.1 Which model gives more readable topics?**

**LDA** gives more readable topics for this corpus. LDA topics are probabilistic distributions where each word is assigned a non-negative weight summing to 1 per topic — this makes top words clearly representative of that topic. LSA produces latent dimensions that can have both positive and negative loadings, so the "top words" of an LSA component are words most associated with one *pole* of a continuous axis; this is less intuitive to read.

**7.2 Which model has more duplicate / noisy / mixed topics?**

- **LSA** has more **mixed topics**: component 0 almost always captures the global mean of the corpus (all three classes), producing a topic that looks like a mix of religious and technical vocabulary.
- Both models produce at least one **generic discourse topic** with Usenet-style vocabulary (think, know, say, people) when k ≥ 5, because the conversational register is shared across all classes.
- **LDA k=8** has more **duplicate topics**: 3 natural themes stretched into 8 components forces splitting of existing topics.

**7.3 Which model is more useful for this corpus specifically?**

**LDA with k=5** is more useful for this 20 Newsgroups corpus. The corpus has 3 natural thematic clusters (atheism, Christianity, electronics), and LDA with k=5 reliably separates electronics from both religious classes (very different vocabulary) and produces at least one topic that partially separates atheism from Christianity (by focusing on argumentation vs. devotional language). LSA struggles with this separation because atheism and Christianity share theological vocabulary — LSA merges them onto a single latent axis. LDA's Dirichlet prior encourages document-to-topic sparsity, making each document "belong" to fewer topics, which better matches the corpus structure where most documents are clearly about one of the three classes. For a classification task, LDA k=5 topics align better with the ground-truth class labels than LSA components do.

In [30]:
# Visualize topic-label alignment for LDA k=5
# For each ground-truth label, show which topic is most dominant
from topic_utils import dominant_topic_per_doc

df_alignment = df_filtered[['category']].copy()
df_alignment['dominant_topic_lda5'] = dominant_topic_per_doc(doc_lda5)
df_alignment['dominant_topic_lsa5'] = dominant_topic_per_doc(doc_lsa5)

print('LDA k=5 — dominant topic distribution per class:')
cross_lda = pd.crosstab(
    df_alignment['category'],
    df_alignment['dominant_topic_lda5'],
    margins=True
)
print(cross_lda.to_string())

print('\nLSA k=5 — dominant topic distribution per class:')
cross_lsa = pd.crosstab(
    df_alignment['category'],
    df_alignment['dominant_topic_lsa5'],
    margins=True
)
print(cross_lsa.to_string())

LDA k=5 — dominant topic distribution per class:
dominant_topic_lda5       0     1    2     3     4   All
category                                                
alt.atheism             653   894  513   105   153  2318
sci.electronics          42    84    4  1760     2  1892
soc.religion.christian  182   600   94    46  1060  1982
All                     877  1578  611  1911  1215  6192

LSA k=5 — dominant topic distribution per class:
dominant_topic_lsa5        0     1    2    3    4   All
category                                               
alt.atheism             1880    12   81  240  105  2318
sci.electronics          676  1190   14    4    8  1892
soc.religion.christian  1910    16    6   44    6  1982
All                     4466  1218  101  288  119  6192


## 12. Generate docs/audit_summary_lab8.md

In [31]:
from topic_utils import generate_audit_md
import os

DOCS_DIR = os.path.join(ROOT, 'docs')
os.makedirs(DOCS_DIR, exist_ok=True)

# ── Identify topic indices from actual model output ──
# After running the models we know the actual topic assignments:
#   LDA k=5:
#     Topic 0: atheism/evidence/belief  → Atheism debate
#     Topic 1: think/people/just/know   → Generic discourse (BAD)
#     Topic 2: islam/islamic/muslim     → Islam/religion subtopic
#     Topic 3: power/ground/use         → Electronics
#     Topic 4: jesus/church/christ/paul → Christianity

def find_topic_by_keywords(topics, keywords):
    """Return index of topic whose top-10 words overlap most with keywords."""
    best_idx, best_score = 0, -1
    for i, words in enumerate(topics):
        score = sum(1 for kw in keywords if kw in words)
        if score > best_score:
            best_score, best_idx = score, i
    return best_idx

christianity_idx = find_topic_by_keywords(words_lda5, ['jesus', 'church', 'christ', 'paul', 'faith', 'lord'])
atheism_idx      = find_topic_by_keywords(words_lda5, ['atheism', 'atheists', 'atheist', 'evidence', 'belief'])
electronics_idx  = find_topic_by_keywords(words_lda5, ['power', 'ground', 'circuit', 'wire', 'voltage'])
islam_idx        = find_topic_by_keywords(words_lda5, ['islam', 'islamic', 'muslim', 'qur', 'rushdie'])
generic_idx      = find_topic_by_keywords(words_lda5, ['think', 'people', 'just', 'like', 'know', 'time'])

print(f'christianity → Topic {christianity_idx}: {words_lda5[christianity_idx]}')
print(f'atheism      → Topic {atheism_idx}: {words_lda5[atheism_idx]}')
print(f'electronics  → Topic {electronics_idx}: {words_lda5[electronics_idx]}')
print(f'islam        → Topic {islam_idx}: {words_lda5[islam_idx]}')
print(f'generic(bad) → Topic {generic_idx}: {words_lda5[generic_idx]}')

best_topics = [
    {
        'name': 'Християнська теологія (LDA k=5, Topic ' + str(christianity_idx) + ')',
        'words': ', '.join(words_lda5[christianity_idx]),
        'why': 'Найчистіша тема: jesus, church, christ, paul, faith, lord. Топ-доки — soc.religion.christian. 1060/1982 документів домінантні тут.',
    },
    {
        'name': 'Атеїзм / суперечки про Бога (LDA k=5, Topic ' + str(atheism_idx) + ')',
        'words': ', '.join(words_lda5[atheism_idx]),
        'why': 'Аргументаційна лексика: atheism, evidence, belief, atheists. Топ-доки — alt.atheism overview і debate posts.',
    },
    {
        'name': 'Іслам / підтема (LDA k=5, Topic ' + str(islam_idx) + ')',
        'words': ', '.join(words_lda5[islam_idx]),
        'why': 'Несподівана реальна підтема: alt.atheism активно обговорював іслам і Рушді.',
    },
    {
        'name': 'Електроніка (LDA k=5, Topic ' + str(electronics_idx) + ')',
        'words': ', '.join(words_lda5[electronics_idx]),
        'why': 'sci.electronics ізольований на 93% (1760/1892 документів → Topic ' + str(electronics_idx) + '). Top words слабкі, але alignment відмінний.',
    },
]

worst_topics = [
    {
        'name': 'Загальний дискурс Usenet (LDA k=5, Topic ' + str(generic_idx) + ')',
        'words': ', '.join(words_lda5[generic_idx]),
        'problem': 'Stop-word / style topic. Поглинає 894 alt.atheism + 600 soc.religion.christian. sklearn English stop-words не покривають Usenet discourse words.',
    },
    {
        'name': 'Username-driven topics (LSA k=5, Topics 2-4)',
        'words': 'bronx, queens, beauchaine, keith, jon, livesey, kent, alink, ksand',
        'problem': 'LSA вловлює патерни активних авторів (Bob Beauchaine .sig, Keith Livesey, Kent/KSAND) замість тем. 3 із 5 LSA компонентів = username artifacts.',
    },
    {
        'name': 'Mixed corpus-wide (LSA k=5, Component 0)',
        'words': 'god, don, people, think, just, know, does, like, say, believe',
        'problem': 'Перший SVD компонент absorbs global variance. Поглинає 1880 alt.atheism + 1910 soc.religion.christian.',
    },
]

audit_results = {
    'corpus_size': len(corpus),
    'min_words': 15,
    'min_df': 5,
    'max_df': 0.90,
    'models_tested': [
        'LSA: TfidfVectorizer (sublinear_tf=True) + TruncatedSVD (k=5, k=8)',
        'LDA: CountVectorizer + LatentDirichletAllocation (max_iter=30) (k=5, k=8)',
    ],
    'k_values': [5, 8],
    'best_topics': best_topics,
    'worst_topics': worst_topics,
    'root_cause': (
        '1. Generic discourse: sklearn English stop-words не покривають Usenet дискурсні слова '
        '(think, know, writes, good). '
        '2. Username artifacts (LSA): активні автори утворюють псевдо-теми. '
        '3. alt.atheism / soc.religion.christian: спільна теологічна лексика — stance невидима для BOW.'
    ),
    'best_model': (
        'LDA k=5 — найкраща модель. Правильно ізолює sci.electronics (93% alignment), '
        'виділяє Christian theology та atheism debate як окремі теми. '
        'LSA: 3/5 компонентів — username artifacts; Component 0 поглинає обидва релігійні класи.'
    ),
    'next_steps': [
        'Додати NEWSGROUP_EXTRA_STOPWORDS: writes, think, know, say, just, good, time, people',
        'Спробувати k=3 (відповідає 3 ground-truth класам) — може дати найчистіші теми',
        'Використати lemma_text із processed_v3 для уніфікації word forms',
        'Додати bigrams для electronics: voltage_divider, power_supply, circuit_board',
        'Фільтрувати username tokens (keith, jon, kent, beauchaine) перед vectorizer',
        'Фільтрувати документи < 50 слів — короткі пости = noise topics',
    ],
}

output_path = os.path.join(DOCS_DIR, 'audit_summary_lab8.md')
generate_audit_md(audit_results, output_path)
print('audit_summary_lab8.md written.')

christianity → Topic 4: ['god', 'jesus', 'church', 'christ', 'sin', 'paul', 'faith', 'love', 'lord', 'man']
atheism      → Topic 0: ['god', 'believe', 'does', 'atheism', 'evidence', 'say', 'don', 'belief', 'atheists', 'true']
electronics  → Topic 3: ['use', 'like', 'used', 'power', 'know', 'just', 'thanks', 'ground', 'don', 'good']
islam        → Topic 2: ['islam', 'religion', 'people', 'islamic', 'writes', 'book', 'jon', 'muslim', 'religious', 'world']
generic(bad) → Topic 1: ['think', 'people', 'don', 'just', 'like', 'know', 'time', 'bible', 'writes', 'good']
Saved: /content/NLP-Lab-works/docs/audit_summary_lab8.md
audit_summary_lab8.md written.


In [32]:
# Coherence note: we compute an approximate c_v coherence using pairwise PMI
# (full gensim coherence requires the raw corpus; we use a lightweight proxy)

def pairwise_cooccurrence_score(topics, tfidf_matrix, vectorizer, n_top=10):
    """
    Proxy coherence: average pairwise cosine similarity between top-word vectors.
    Higher = words appear in similar contexts = more coherent topic.
    Not identical to c_v but directionally correct.
    """
    from sklearn.metrics.pairwise import cosine_similarity
    vocab = vectorizer.vocabulary_
    X = tfidf_matrix.toarray()
    scores = []
    for words in topics:
        indices = [vocab.get(w, -1) for w in words[:n_top]]
        indices = [i for i in indices if i >= 0]
        if len(indices) < 2:
            scores.append(0.0)
            continue
        word_vecs = X[:, indices].T  # shape: (n_words, n_docs)
        sim_matrix = cosine_similarity(word_vecs)
        upper = sim_matrix[np.triu_indices(len(indices), k=1)]
        scores.append(float(np.mean(upper)))
    return scores

coh_lsa5 = pairwise_cooccurrence_score(words_lsa5, tfidf5, vec_tfidf5)
coh_lda5 = pairwise_cooccurrence_score(words_lda5, tfidf5, vec_tfidf5)

print('Proxy coherence scores (pairwise cosine similarity of top-word vectors):')
print('NOTE: This is a proxy, not c_v. Higher = more coherent topic vocabulary.')
print(f'\nLSA k=5:')
for i, s in enumerate(coh_lsa5):
    print(f'  Topic {i}: {s:.4f}  | {words_lsa5[i]}')
print(f'  Mean: {np.mean(coh_lsa5):.4f}')

print(f'\nLDA k=5:')
for i, s in enumerate(coh_lda5):
    print(f'  Topic {i}: {s:.4f}  | {words_lda5[i]}')
print(f'  Mean: {np.mean(coh_lda5):.4f}')

print('\nIMPORTANT: Coherence is a supplementary signal only.')
print('Manual interpretation + top document review are the primary evaluation.')

Proxy coherence scores (pairwise cosine similarity of top-word vectors):
NOTE: This is a proxy, not c_v. Higher = more coherent topic vocabulary.

LSA k=5:
  Topic 0: 0.2671  | ['god', 'don', 'people', 'think', 'just', 'know', 'does', 'like', 'say', 'believe']
  Topic 1: 0.0861  | ['god', 'thanks', 'use', 'chip', 'jesus', 'circuit', 'believe', 'mail', 'output', 'voltage']
  Topic 2: 0.8050  | ['bronx', 'sank', 'queens', 'manhattan', 'blew', 'beauchaine', 'bob', 'sea', 'stay', 'away']
  Topic 3: 0.1649  | ['keith', 'jon', 'morality', 'writes', 'livesey', 'god', 'objective', 'jesus', 'moral', 'christ']
  Topic 4: 0.4197  | ['kent', 'alink', 'ksand', 'private', 'activities', 'cheers', 'net', 'wrote', 'jon', 'writes']
  Mean: 0.3486

LDA k=5:
  Topic 0: 0.1883  | ['god', 'believe', 'does', 'atheism', 'evidence', 'say', 'don', 'belief', 'atheists', 'true']
  Topic 1: 0.2281  | ['think', 'people', 'don', 'just', 'like', 'know', 'time', 'bible', 'writes', 'good']
  Topic 2: 0.1170  | ['islam'

In [33]:
print('\n' + '='*60)
print('Lab 8 complete.')
print(f'Corpus size: {len(corpus):,} documents')
print('Models: LSA k=5, LSA k=8, LDA k=5, LDA k=8')
print('Outputs written:')
print(f'  docs/audit_summary_lab8.md')
print('='*60)


Lab 8 complete.
Corpus size: 6,192 documents
Models: LSA k=5, LSA k=8, LDA k=5, LDA k=8
Outputs written:
  docs/audit_summary_lab8.md
